# Extract DINOv3 features from the full Causal-VidQA MP4 dataset

Notebook này chỉ làm một việc: lấy đều 16 frame/video, chạy DINOv3 ViT-L/16 và lưu một tensor `[16, 1024]` cho mỗi video.

- Nguồn video: `danielq07/causal-vidqa-raw-video-full` qua `kagglehub.dataset_download`.
- Không dùng split pickle, không tải part phụ, không ghép video và không upload Hugging Face.
- Output phẳng: `/kaggle/working/dinov3_features/<video_id>.pt`.
- Hãy bật **GPU Accelerator**. Model DINOv3 có thể yêu cầu chấp nhận điều khoản trên Hugging Face và Kaggle Secret `HF_TOKEN`.


In [ ]:
# Cài dependency riêng cho notebook extract; restart session nếu Kaggle yêu cầu.
%pip install -q -U "kagglehub>=1,<2" "transformers>=4.56,<5" "decord>=0.6,<1" "tqdm>=4.64,<5"


In [ ]:
import os
from pathlib import Path
from collections import Counter

import kagglehub
import numpy as np
import torch
from decord import VideoReader, cpu
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel


def read_kaggle_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)


HF_TOKEN = read_kaggle_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
RAW_DATASET = "danielq07/causal-vidqa-raw-video-full"
OUTPUT_DIR = Path("/kaggle/working/dinov3_features")
N_FRAMES = 16
FEATURE_DIM = 1024
USE_AMP = True

if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa được bật. Vào Kaggle Settings -> Accelerator -> GPU rồi chạy lại.")

device = torch.device("cuda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dataset_root = Path(kagglehub.dataset_download(RAW_DATASET))
video_files = sorted(dataset_root.rglob("*.mp4"))
if not video_files:
    raise FileNotFoundError(f"Không tìm thấy MP4 trong {dataset_root}")

stems = [p.stem for p in video_files]
duplicates = sorted(stem for stem, count in Counter(stems).items() if count > 1)
if duplicates:
    raise RuntimeError(f"Trùng video_id (tên file không kèm .mp4): {duplicates[:10]}")

print("Dataset root:", dataset_root)
print("Số MP4:", len(video_files))
print("Output:", OUTPUT_DIR)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
print("Đang tải DINOv3:", MODEL_ID)
processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModel.from_pretrained(MODEL_ID, token=HF_TOKEN)
model.eval().to(device)


def uniform_indices(total_frames, count=N_FRAMES):
    if total_frames <= 0:
        raise ValueError("Video không có frame")
    return np.linspace(0, total_frames - 1, count).round().astype(np.int64)


def extract_one(video_path):
    reader = VideoReader(str(video_path), ctx=cpu(0), num_threads=1)
    indices = uniform_indices(len(reader))
    frames = reader.get_batch(indices).asnumpy()  # RGB uint8, [T,H,W,3]
    inputs = processor(images=list(frames), return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device, non_blocking=True)

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            outputs = model(pixel_values=pixel_values)
            features = getattr(outputs, "pooler_output", None)
            if features is None:
                features = outputs.last_hidden_state[:, 0]

    features = features.float().cpu().contiguous()
    expected = (N_FRAMES, FEATURE_DIM)
    if tuple(features.shape) != expected:
        raise RuntimeError(f"Sai shape {tuple(features.shape)}, cần {expected}")
    if not torch.isfinite(features).all():
        raise RuntimeError("Feature có NaN hoặc Inf")
    return features


def valid_saved_feature(path):
    if not path.is_file():
        return False
    try:
        value = torch.load(path, map_location="cpu", weights_only=True)
        return isinstance(value, torch.Tensor) and tuple(value.shape) == (N_FRAMES, FEATURE_DIM) and torch.isfinite(value).all()
    except Exception:
        return False


print("Model đã sẵn sàng")


## Smoke test

Cell này giải mã và extract một video trước. Chỉ chạy full dataset nếu shape là `(16, 1024)` và không có NaN/Inf.


In [ ]:
sample_path = video_files[0]
sample_feature = extract_one(sample_path)
print("Video:", sample_path.name)
print("Shape:", tuple(sample_feature.shape))
print("dtype:", sample_feature.dtype)
print("finite:", bool(torch.isfinite(sample_feature).all()))


## Extract toàn bộ MP4

Mỗi file được ghi tạm rồi đổi tên để tránh giữ file hỏng nếu runtime dừng giữa chừng. Khi chạy lại, feature hợp lệ đã có sẽ được bỏ qua. Lỗi được lưu tại `/kaggle/working/dinov3_extraction_errors.csv`.


In [ ]:
import csv

errors = []
saved = 0
skipped = 0

for video_path in tqdm(video_files, desc="DINOv3 videos"):
    output_path = OUTPUT_DIR / f"{video_path.stem}.pt"
    if valid_saved_feature(output_path):
        skipped += 1
        continue

    temp_path = output_path.with_suffix(".pt.tmp")
    try:
        feature = extract_one(video_path)
        torch.save(feature, temp_path)
        temp_path.replace(output_path)
        saved += 1
    except Exception as exc:
        if temp_path.exists():
            temp_path.unlink()
        errors.append((video_path.stem, str(video_path), repr(exc)))

error_csv = Path("/kaggle/working/dinov3_extraction_errors.csv")
with error_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["video_id", "video_path", "error"])
    writer.writerows(errors)

print(f"Mới lưu: {saved} | Bỏ qua hợp lệ: {skipped} | Lỗi: {len(errors)}")
print("Error report:", error_csv)


In [ ]:
feature_paths = sorted(OUTPUT_DIR.glob("*.pt"))
bad = []
for path in tqdm(feature_paths, desc="Verify saved features"):
    if not valid_saved_feature(path):
        bad.append(path.name)

print("MP4 đầu vào:", len(video_files))
print("Feature hợp lệ:", len(feature_paths) - len(bad))
print("Feature lỗi:", len(bad))
print("Video extract lỗi:", len(errors))

assert not bad, f"Có feature sai: {bad[:10]}"
assert len(feature_paths) + len(errors) == len(video_files), (
    "Số output + lỗi không khớp số video; kiểm tra duplicate hoặc runtime bị dừng."
)

if errors:
    print("Chưa hoàn tất. Xem CSV lỗi rồi chạy lại cell extract.")
else:
    print("Hoàn tất. Save Version để lưu thư mục dinov3_features trong Kaggle Output.")
